# Parallel Processing Fundamentals
## *Prof. Gary L. Pavlis*

Before we consider seismic processing with MsPASS and dask, I think it will be helpful to look at some simpler examples.   Parallel processing with dask is, itself, a large subject.  I focus here on fundamentals you will need to understand in order to comprehend the more complicated seismic processing examples we will examine later in this session.

Concepts this tutorial aims to help you understand are:
1.  Schedulers and workers and how they interact
2.  The dask bag, which we will use for seismic processing with dask, can thought of as a list of things that do not necessarily fit in memory.
3.  Dask uses lazy/delayed computation.  That means you construct the processing sequence before you actually start it.
4.  The bag map operator applies a function with optional arguments to modify each element in the bag. 

First, we need this little incantation to connect to the dask scheduler that is already running on geolab.   Note if you are reading this outside of the MsPASS short course, this notebook cannot be run without modification outside GeoLab.

In [ ]:
from mspasspy.client import Client
mspass_client = Client()   
db = mspass_client.get_database("ES2026")

Now run this line.  Click on the hyperlink after the "Dashboard:" tag:

In [ ]:
dask_client = mspass_client.get_scheduler()
dask_client

When you clicked on that link you should now have another tab in your display with the label "Dask: Status".   That provides a window into cluster operations that can help you understand what is going on below.   Keep that tab open as we'll be looking at it repeatedly.

## Trivial example
The first does a trivial calculation but provides a simple example for how to use dask to run a loop in parallel.   I emphasize the calculation is trivial and is not something you would normally need to run in parallel.   It  teaches some additional concepts that should be useful for later.  
1.  The works by harvesting data from the MongoDB database.
2.  It provides the simplest example I can think of to show how a loop over database documents can be transformed into a parallel job.

The actual thing this example does is compute the time difference between end time computed from the sample rate and number of samples to the "endtime" attribute stored in the MongoDB database.   There are cases where that calculation is useful when dealing with timing problems, but all the data here, in fact, have no such issues.  The point is that the calculation is trivial but is I hope it will be educational for you.

First, define a couple of functions.  They aren't really essential, but are helpful to teach the concepts here.  

In [ ]:
def extract_tuple(doc):
    """
    Toy function for bag map illustration.  Extracts attributes "starttime", "endtime", "delta",
    and "npts" and returns them in a tuple in that order. 

    Point is to illustrate the content of a bag can change after a map is applied.
    """
    return [doc["starttime"],doc["endtime"],doc["delta"],doc["npts"]]
def endtime_error(time_tuple):
    """
    Takes the output of the function above and computes the time mismatch between 
    the tabulated endtime and that computed from starttime + delta*npts.
    """
    # not necessary to create these temporaries but clearer for a tutorial like this
    stime=time_tuple[0]
    etime=time_tuple[1]
    dt = time_tuple[2]
    npts = time_tuple[3]
    error = etime - (stime + dt*(npts-1))
    return error

In [ ]:
# serial version
import time 
t0 = time.time()
err_list=list()
ndocs=db.wf_Seismogram.count_documents({})
cursor = db.wf_Seismogram.find({})
for doc in cursor:
    tup = extract_tuple(doc)
    err = endtime_error(tup)
    err_list.append(err)
t = time.time()
print("Processed ",ndocs," items in ",t-t0," seconds")

Look at the output and we see we have an all clear result - all the computed times were 0

In [ ]:
print(min(err_list),max(err_list))

The above is model for serial seismic processing.  The processing is driven by a loop over database documents returned by the cursor returned by find.  (recall that concept from Session1).  A cursor is an iterable designed to operate on huge lists that are retrieved in blocks from the database server.  For parallel jobs, we will want to convert that into a simple list.  This next box does that to create a list of dictionaries (documents):

In [ ]:
cursor=db.wf_Seismogram.find({})
print("The type of cursor=",type(cursor))
doclist=list(cursor)
print("The type of doclist=",type(doclist))
print("The list size = ",len(doclist))

The doclist content will drive this processing.   The two functions will be run on each item in the list.   Here is the syntax of the parallel version of the trivial serial loop we did above.

In [ ]:
import dask.bag as dbg
mybag = dbg.from_sequence(doclist)
print("The type of the symbol mybag=",type(mybag))
mybag = mybag.map(extract_tuple)
mybag = mybag.map(endtime_error)
print("The type of the symbol mybag after two map calls=",type(mybag))

Notice:
- We create the thing called a bag from doclist using the "from_sequence" function.
- We define the processing sequence to run the two functions "extract_tuple" and "endtime_error" in that order on each item in the bag.
- Look at the dask diagnostics window - nothing has happened yet.
- The type of "mybag" doesn't change.

This illustrates the idea of a delayed/lazy calculation.   mybag is just a skeleton (the DAG I discussed in lecture) that the scheduler will use to process the bag content in parallel.  

With that background, run this next box and watch what happens in the dask diagnostics window.

In [ ]:
t0=time.time()
err_list_parallel=mybag.compute()
t=time.time()
print("Size of result=",len(err_list))
print("Time to run=",t-t0)

Note we got the same answer but it took a lot longer because this silly example is not appropriate for parallelization.  The parallel overhead makes it run significantly slower than the serial one.  This, however, shows we do get the same answer.

In [ ]:
print(type(err_list_parallel))
print(len(err_list_parallel))
print(max(err_list_parallel))
print(min(err_list_parallel))

## More Complicated Example


In [ ]:
import numpy as np
def make_random_vector(float_seed,N,loc=0.0,scale=1.0,)->np.ndarray:
    """
    Create a vector of length N of Gaussian distributed noise with sigma=scale and mean=loc.   
    arg0 (float_seed) is a floating point number to use as a seed for the random number 
    generator.  In this tutorial it is itself generated from a random number generator
    using a particular seed (42).  As the name suggests it is expected to be a float but 
    it is truncated to an int that is used for the randoms eed.
    """
    rng = np.random.default_rng(seed = int(float_seed))  
    d = rng.normal(loc=loc,scale=scale,size=N)
    return d
def sumsq(d):
    """
    Return sum of squares of input numpy vector d.   Numpy doesn't have an explicit function 
    for this but what we use here is vectorized and very fast.
    """
    return np.sum(d*d)

In [ ]:
import dask.bag as dbg
# 42 is the secret to life, the universe, and everything (Hitchhiker's guide to the Galaxy)
rng = np.random.default_rng(seed=42) 
# generate a list of 100000 random number generator seeds
seeds = rng.uniform(low=0.0, high=1000000, size=100000)

In [ ]:
Nsamp = 1000   # number of samples per vector
t0=time.time()
mybag = dbg.from_sequence(seeds,npartitions=20)
mybag = mybag.map(make_random_vector,Nsamp)
mybag = mybag.map(sumsq)
result = mybag.compute()
t=time.time()
print("Execution time=",t-t0)

In [ ]:
import matplotlib.pyplot as plt
print("Size of result vector=",len(result))
plt.hist(result,bins=100)